# Practical 5: Neural Networks
## Building a Multi-Layer Perceptron (MLP) in PyTorch to classify wheat seed samples

Welcome! This notebook is a **guided walkthrough** of the assignment. Every bullet-point requirement is
implemented below and clearly labelled. Cells marked **✏️ YOUR TURN** are yours to fill in — the notebook
explains *why* before you *do*.

### The dataset: UCI "Seeds"
Measurements of geometric properties of wheat kernels belonging to **three varieties**: *Kama*, *Rosa*,
*Canadian*. There are **210 samples** (70 per class) and **7 real-valued features**:
`area`, `perimeter`, `compactness`, `length of kernel`, `width of kernel`, `asymmetry coefficient`,
`length of kernel groove`.

This is a small, *tabular* (not image) dataset — perfect for a small MLP.

> **Citation:** Charytanowicz, M., Niewczas, J., Kulczycki, P., Kowalski, P., & Lukasik, S. (2010).
> *Seeds* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5H30K

### What we must implement (checklist)
- [x] Custom `nn.Module` subclass
- [x] `forward()` method
- [x] A `Dataset` **and** a `DataLoader`
- [x] Mini-batch training
- [x] A **hand-written loss function**, normalised by the number of inputs, accumulated over mini-batches
- [x] An optimizer
- [x] Batch Normalization
- [x] Training loop using `model.train()`
- [x] Evaluation loop using `model.eval()`
- [x] Save/load weights with `state_dict`
- [x] Loss logging + live visualisation (TensorBoard, plus an inline fallback)
- [x] Hyperparameter investigation
- [x] At least one regularization technique, compared to baseline
- [x] Write `network.csv` with final test predictions

---
## 0. Setup and imports

We start by importing PyTorch and friends, fixing a **random seed** so our experiments are reproducible,
and choosing a **device** (GPU if available, otherwise CPU).

> **Note on devices:** the assignment explicitly asks us to move data and model to the appropriate device.
> We define `DEVICE` once and use `.to(DEVICE)` everywhere.

In [ ]:
import os
import copy
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# sklearn is only used for data loading (fetch_ucirepo / CSV fallback) and splitting,
# NOT for the model itself. We implement the network ourselves.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ----------------------------------------------------------------
# Reproducibility: one seed to rule them all
# ----------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# Device selection
# ----------------------------------------------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():   # Apple Silicon
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {DEVICE}")

### Try to launch TensorBoard

The assignment *hints* at TensorBoard. It only works if the `tensorboard` package is installed.
We'll detect it here; if it's missing, we fall back to a small inline matplotlib logger that updates the
loss curve live in the notebook. Either way you get **live loss visualisation**.

In [ ]:
# Is TensorBoard available?
try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
    print("TensorBoard is available ✔")
    print("To view it, run in a terminal:  tensorboard --logdir runs")
    print("Then open the URL it prints (usually http://localhost:6006).")
except ImportError:
    HAS_TENSORBOARD = False
    print("TensorBoard not installed — will use the inline matplotlib logger instead.")
    print("Install it later with:  pip install tensorboard")

---
## 1. Load the Seeds data

We load the dataset as a pandas DataFrame. Two strategies, in order:
1. `ucimlrepo` (official download, returns features + targets).
2. A local `seeds_dataset.txt` / `seeds.csv` file if you already have it.

The features are physically different scales (area ≈ 15, compactness ≈ 0.9...). Neural networks *hate*
un-scaled inputs, so we **standardise** them (zero mean, unit variance). Crucially, we fit the scaler on
the **training set only** to avoid information leaking from the test set.

In [ ]:
FEATURE_NAMES = [
    "area", "perimeter", "compactness",
    "length_kernel", "width_kernel",
    "asymmetry_coeff", "length_groove",
]

def load_seeds():
    """Return (X: np.ndarray[n,7], y: np.ndarray[n] in {0,1,2})."""
    # --- Strategy 1: official UCI download ---
    try:
        from ucimlrepo import fetch_ucirepo
        seeds = fetch_ucirepo(id=236)
        X = seeds.data.features.to_numpy(dtype=np.float32)
        # UCI encodes classes as 1,2,3 -> shift to 0,1,2 for PyTorch
        y = seeds.data.targets.to_numpy().ravel().astype(np.int64) - 1
        print("Loaded Seeds from UCI repository.")
        return X, y
    except Exception as e:
        print(f"ucimlrepo fetch failed ({e}); trying local files...")

    # --- Strategy 2: local file ---
    for path in ["seeds_dataset.txt", "seeds.csv", "data/seeds_dataset.txt", "data/seeds.csv"]:
        if Path(path).exists():
            sep = r"\s+" if path.endswith(".txt") else ","
            df = pd.read_csv(path, sep=sep, header=None)
            X = df.iloc[:, :7].to_numpy(dtype=np.float32)
            y = df.iloc[:, 7].to_numpy().astype(np.int64) - 1
            print(f"Loaded Seeds from {path}")
            return X, y

    raise FileNotFoundError(
        "Could not find the Seeds data. Run `pip install ucimlrepo`, or download "
        "seeds_dataset.txt from https://archive.ics.uci.edu/dataset/236/seeds"
    )

X, y = load_seeds()
print("Feature matrix:", X.shape, "| labels:", np.bincount(y), "(class counts)")
X[:3]

### Train / validation / test split

We use an **80 / 10 / 10** split (train / val / test):
- **train** — weights are updated on this.
- **validation** — used to monitor overfitting, tune hyperparameters, and drive early stopping.
- **test** — touched **once**, at the very end, for the `network.csv` predictions.

Standardisation is fit on the training partition only.

In [ ]:
# 80 / 10 / 10 split. Stratify keeps class balance across splits.
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp
)

print(f"train: {len(y_train)}   val: {len(y_val)}   test: {len(y_test)}")

# Standardise: FIT on train, TRANSFORM all three splits.
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

print("Post-scaling feature means (≈0):", np.round(X_train.mean(axis=0), 3))
print("Post-scaling feature stds  (≈1):", np.round(X_train.std(axis=0), 3))

N_FEATURES = X_train.shape[1]
N_CLASSES  = len(np.unique(y))

---
## 2. A custom `Dataset` and `DataLoader`

**Requirement: a Dataset and a DataLoader.**

A `Dataset` is a tiny class that answers two questions: *"how many items are there?"* (`__len__`) and
*"give me item `i`"* (`__getitem__`). The `DataLoader` then does the heavy lifting: batching, shuffling,
and (optionally) parallel loading.

**Important gotcha:** `nn.CrossEntropyLoss` (and our own cross-entropy) expects *integer class indices*
as targets — shape `(batch,)` — not one-hot vectors. And every tensor fed to a `nn.Linear` must be
`float32`. Our `__getitem__` guarantees both.

In [ ]:
class SeedsDataset(Dataset):
    """Wraps numpy arrays as a PyTorch Dataset.

    Each item is (features, label):
      features -> float32 tensor of shape (7,)
      label    -> int64   tensor of shape ()   (class index in {0,1,2})
    """

    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Store as torch tensors up-front (cheap for a small dataset).
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]


# Build the three datasets
train_ds = SeedsDataset(X_train, y_train)
val_ds   = SeedsDataset(X_val,   y_val)
test_ds  = SeedsDataset(X_test,  y_test)

# Build the DataLoaders.
#   - train: SHUFFLE (order matters — we want decorrelated mini-batches)
#   - val/test: no shuffle (deterministic evaluation)
#   - num_workers=0 keeps things simple & notebook-safe
BATCH_SIZE = 16  # a hyperparameter we revisit later

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check: peek at one mini-batch
xb, yb = next(iter(train_loader))
print("batch features:", xb.shape, xb.dtype)
print("batch labels  :", yb.shape, yb.dtype, "->", yb[:5].tolist())

---
## 3. The network architecture

**Requirement: 1 input layer, 1 hidden layer, 1 output layer — a custom `nn.Module` with a `forward()`.**

```
  input (7 features)
        │
   Linear(7 → H)          ← "input → hidden" connection
        │
   BatchNorm1d(H)         ← requirement: Batch Normalization
        │
   Activation             ← hyperparameter: ReLU / LeakyReLU / Tanh
        │
   Dropout(p)             ← optional regularization (can be disabled)
        │
   Linear(H → 3)          ← output layer: one raw score (logit) per class
```

To let us *investigate hyperparameters* (number of hidden layers, neurons, activation, dropout), we build
the network **generically** through `__init__` arguments. The special case `hidden_sizes=(H,)` gives exactly
the required three-layer architecture.

### Design notes
- We output **logits** (no softmax at the end). Our custom loss applies `log_softmax` internally, which is
  numerically stabler than `softmax` followed by `log`.
- `nn.BatchNorm1d` for tabular data is `BatchNorm1d` (not `2d` — no height/width dims).
- The activation is looked up from a dict so we can swap it by *name* (great for the hyperparameter sweep).

In [ ]:
# Map a name -> activation module. Swap freely during the hyperparameter study.
def get_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    elif name == "leakyrelu":
        return nn.LeakyReLU(negative_slope=0.01)
    elif name == "tanh":
        return nn.Tanh()
    elif name == "sigmoid":
        return nn.Sigmoid()
    else:
        raise ValueError(f"Unknown activation: {name}")


class MLP(nn.Module):
    """A general fully-connected MLP built from scratch (no pre-built model imported).

    Args:
        in_features:    number of input features (7 for Seeds)
        hidden_sizes:   tuple of hidden-layer widths, e.g. (16,) or (32, 16)
        n_classes:      number of output classes (3)
        activation:     'relu' | 'leakyrelu' | 'tanh' | 'sigmoid'
        dropout:        dropout probability (0.0 disables it)
        use_batchnorm:  whether to insert BatchNorm1d after each hidden Linear layer
    """

    def __init__(
        self,
        in_features: int,
        hidden_sizes=(16,),
        n_classes: int = 3,
        activation: str = "relu",
        dropout: float = 0.0,
        use_batchnorm: bool = True,
    ):
        super().__init__()

        layers = []
        prev = in_features

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))          # affine transform
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))       # normalise pre-activations
            layers.append(get_activation(activation))  # non-linearity
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))     # regularization
            prev = h

        layers.append(nn.Linear(prev, n_classes))      # output layer -> logits

        # nn.Sequential is just an ordered container; we still wrote the architecture ourselves.
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: (batch, in_features) -> (batch, n_classes) raw logits."""
        return self.net(x)


# Smoke test: instantiate the required 3-layer network and check the output shape.
model = MLP(in_features=N_FEATURES, hidden_sizes=(16,), n_classes=N_CLASSES,
            activation="relu", dropout=0.0, use_batchnorm=True).to(DEVICE)

dummy = torch.randn(5, N_FEATURES, device=DEVICE)
print(model)
print("Output shape for a batch of 5:", model(dummy).shape)   # expect (5, 3)

---
## 4. A hand-written loss function

**Requirement: code the loss yourself. Normalise it by the input values. Accumulate over mini-batches.**

We implement **cross-entropy loss from scratch**, i.e. the negative log-likelihood of the correct class
under a softmax:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log\Big(\frac{\exp(z_{i,c_i})}{\sum_{k}\exp(z_{i,k})}\Big)$$

where $z$ are the logits and $c_i$ is the true class of example $i$.

### Two points the assignment is testing
1. **Normalise by the input values** — divide the summed loss by the number of *examples* in the batch
   (`N`), so the number you log is a **per-example average** that is comparable across batches of
   different sizes. (Note: we normalise per-example; summing per-example losses and dividing by `N`
   is exactly what `reduction='mean'` does.)
2. **Accumulate over mini-batches** — we return the mean loss *and* the batch size, so the training loop
   can weight each batch and compute a correct running average over the epoch.

### Numerical stability
Naively computing `log(softmax(z))` overflows for large `z`. We use the **log-sum-exp trick**:
subtract the max logit before exponentiating, then add it back. This is mathematically identical but
stable, and it is exactly the trick behind `nn.CrossEntropyLoss`.

> We use `torch.autograd`-compatible ops throughout, so `loss.backward()` just works on our hand-coded loss.

In [ ]:
def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Cross-entropy loss, computed from scratch.

    Args:
        logits:  (N, C) raw, unnormalised scores.
        targets: (N,) integer class indices in [0, C).
    Returns:
        Scalar tensor = mean per-example negative log-likelihood.
    """
    N = logits.shape[0]                       # number of examples in the batch

    # --- log-sum-exp trick for stability ---
    max_logits = logits.max(dim=1, keepdim=True).values      # (N, 1)
    shifted = logits - max_logits                            # (N, C)
    logsumexp = max_logits.squeeze(1) + torch.log(torch.exp(shifted).sum(dim=1))  # (N,)

    # log-softmax of the correct class: z_correct - logsumexp
    # gather picks, for each row, the logit at its target index.
    correct_logits = logits.gather(1, targets.unsqueeze(1)).squeeze(1)   # (N,)
    log_probs = correct_logits - logsumexp                               # (N,) = log p(y|x)

    # Negative log-likelihood, NORMALISED by N (number of inputs).
    loss = -log_probs.sum() / N
    return loss, N                            # return N too, for correct accumulation


# --- Verify against PyTorch's built-in CrossEntropyLoss (sanity check only) ---
torch.manual_seed(0)
logits = torch.randn(8, 3, dtype=torch.float32)
targets = torch.randint(0, 3, (8,))

ours, _ = cross_entropy_loss(logits, targets)
reference = nn.CrossEntropyLoss()(logits, targets)
print(f"our loss       : {ours.item():.6f}")
print(f"PyTorch's loss : {reference.item():.6f}")
print("match:", torch.allclose(ours, reference, atol=1e-6))

---
## 5. Live loss logging

We build a small `Logger` that:
- writes scalars to **TensorBoard** when available (`add_scalar`), and
- falls back to an **inline matplotlib** plot that we redraw periodically so you can *watch* the losses
  move while the network trains.

We log three curves: **train loss**, **validation loss**, and **validation accuracy**.

In [ ]:
class Logger:
    """Logs losses to TensorBoard (if present) and to an inline matplotlib plot."""

    def __init__(self, run_name: str = "run", use_tensorboard: bool = HAS_TENSORBOARD):
        self.history = {"train_loss": [], "val_loss": [], "val_acc": []}
        self.run_name = run_name
        self.use_tb = use_tensorboard
        self.writer = None
        self.log_dir = None

        if self.use_tb:
            # Each run gets its own subdirectory so multiple runs can be compared.
            self.log_dir = f"runs/{run_name}"
            # (Re-)create the writer at each run; timestamps make comparison easy.
            from datetime import datetime
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            self.writer = SummaryWriter(f"{self.log_dir}_{ts}")

    def log(self, epoch: int, train_loss: float, val_loss: float, val_acc: float):
        self.history["train_loss"].append(train_loss)
        self.history["val_loss"].append(val_loss)
        self.history["val_acc"].append(val_acc)
        if self.writer is not None:
            self.writer.add_scalar("Loss/train", train_loss, epoch)
            self.writer.add_scalar("Loss/val",   val_loss,   epoch)
            self.writer.add_scalar("Accuracy/val", val_acc,  epoch)

    def flush(self):
        if self.writer is not None:
            self.writer.flush()

    def close(self):
        if self.writer is not None:
            self.writer.close()

    def plot(self, title: str = ""):
        """Redraw the inline loss/accuracy curve. Call this every few epochs."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
        ep = range(1, len(self.history["train_loss"]) + 1)
        ax1.plot(ep, self.history["train_loss"], label="train")
        ax1.plot(ep, self.history["val_loss"],   label="validation")
        ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend()
        ax1.set_title(f"Loss — {title or self.run_name}")
        ax2.plot(ep, self.history["val_acc"], color="tab:green", label="val acc")
        ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.legend()
        ax2.set_title("Validation accuracy")
        ax2.set_ylim(0, 1.05)
        plt.tight_layout(); plt.show()


def enable_live_plot():
    """Use interactive mode so the plot updates in-place while training."""
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            get_ipython().run_line_magic("matplotlib", "inline")
    except Exception:
        pass

print("Logger ready.")

---
## 6. Training and evaluation loops

**Requirements: mini-batch training loop with `model.train()`; evaluation loop with `model.eval()`.**

### Why `model.train()` vs `model.eval()` matters
These flags change the behaviour of two layer types:
- **Dropout** — active (drops neurons) in `train()`, identity in `eval()`.
- **BatchNorm** — uses *batch* statistics and updates its running estimates in `train()`; uses the
  frozen *running* statistics in `eval()`.

Forgetting `model.eval()` is the #1 source of "my validation loss looks worse than training" bugs.

### The training epoch, step by step
For each mini-batch:
1. `optimizer.zero_grad()` — clear gradients from the previous step.
2. Move the batch to the device.
3. `logits = model(xb)` — forward pass.
4. `loss, n = cross_entropy_loss(logits, yb)` — our hand-written loss.
5. `loss.backward()` — backprop.
6. `optimizer.step()` — update weights.
7. **Accumulate** `loss.item() * n` and `n`, then divide at the end for a size-weighted epoch average.

> **Accumulation detail:** we accumulate `loss * n` and total `n`, then divide. This is slightly different
> from just averaging per-batch means, and it's the *correct* way: it weights each batch by how many
> examples it contained (the last batch may be smaller).

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    """Run the model over a loader in EVAL mode.

    Returns:
        avg_loss: size-weighted mean loss over the loader
        accuracy: fraction of correctly classified examples
    """
    model.eval()                    # <- disables dropout, uses running BatchNorm stats
    total_loss = 0.0
    total_n = 0
    correct = 0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss, n = cross_entropy_loss(logits, yb)

        total_loss += loss.item() * n          # weight by batch size for a correct average
        total_n += n
        correct += (logits.argmax(dim=1) == yb).sum().item()

    return total_loss / total_n, correct / total_n


def train(model, train_loader, val_loader, optimizer, n_epochs=60,
          logger=None, early_stopping_patience=None, min_delta=0.0,
          scheduler=None, plot_every=10, verbose=True):
    """Full training loop.

    Implements: mini-batch training, loss accumulation, eval loop, optional early stopping.
    Saves the BEST validation-loss weights into the model (and returns the history).
    """
    if logger is None:
        logger = Logger()

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())   # snapshot of best weights
    epochs_without_improvement = 0

    for epoch in range(1, n_epochs + 1):
        # ----------------------------- TRAIN -----------------------------
        model.train()                       # <- enable dropout + BatchNorm batch stats
        running_loss = 0.0
        running_n = 0

        for xb, yb in train_loader:         # mini-batch iteration
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            optimizer.zero_grad()           # 1. clear old gradients
            logits = model(xb)              # 2. forward pass
            loss, n = cross_entropy_loss(logits, yb)   # 3. our hand-written loss
            loss.backward()                 # 4. backprop
            optimizer.step()                # 5. update weights

            # Accumulate loss *normalised by the number of inputs*, weighted by batch size.
            running_loss += loss.item() * n
            running_n += n

        train_loss = running_loss / running_n   # size-weighted average over the epoch

        # --------------------------- VALIDATE ---------------------------
        val_loss, val_acc = evaluate(model, val_loader)

        # ---------------------------- LOG -------------------------------
        logger.log(epoch, train_loss, val_loss, val_acc)
        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"epoch {epoch:3d} | train loss {train_loss:.4f} "
                  f"| val loss {val_loss:.4f} | val acc {val_acc:.3f}")

        # ----------------------- LIVE VISUALISATION --------------------
        if plot_every and epoch % plot_every == 0:
            logger.plot()
            logger.flush()

        # --------------------------- SCHEDULER -------------------------
        if scheduler is not None:
            scheduler.step()

        # ---------------------- EARLY STOPPING -------------------------
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())   # keep best weights
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if early_stopping_patience and epochs_without_improvement >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch} "
                      f"(no improvement for {early_stopping_patience} epochs).")
                break

    # Restore the best weights seen during training.
    model.load_state_dict(best_state)
    logger.history["best_val_loss"] = best_val_loss
    return model, logger.history

---
## 7. Baseline training run

Now we put it together: the **required 3-layer architecture** (7 → 16 → 3), BatchNorm, ReLU, **Adam**
optimizer, mini-batches of 16, and our hand-written loss. No regularization yet — this is our **baseline**.

Neural networks need a clean slate per run, so we define a `make_model` helper that re-seeds and rebuilds
everything.

In [ ]:
def make_model(hidden_sizes=(16,), activation="relu", dropout=0.0,
               use_batchnorm=True, weight_decay=0.0, lr=1e-3):
    """Re-seed, build a fresh model on DEVICE, and return (model, optimizer)."""
    torch.manual_seed(SEED)
    model = MLP(in_features=N_FEATURES, hidden_sizes=hidden_sizes,
                n_classes=N_CLASSES, activation=activation,
                dropout=dropout, use_batchnorm=use_batchnorm).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    return model, optimizer


# ---------------------------- BASELINE RUN ----------------------------
enable_live_plot()

base_model, base_opt = make_model(hidden_sizes=(16,), activation="relu", lr=1e-3)
base_logger = Logger(run_name="baseline")

base_model, base_hist = train(
    base_model, train_loader, val_loader, base_opt,
    n_epochs=60, logger=base_logger, early_stopping_patience=15, plot_every=15,
)

base_train_acc = evaluate(base_model, train_loader)[1]
base_val_loss, base_val_acc = evaluate(base_model, val_loader)
print(f"\nBASELINE -> train acc {base_train_acc:.3f} | val acc {base_val_acc:.3f}")

---
## 8. Save & load with `state_dict`

**Requirement: saving and loading model parameters using `state_dict`.**

`state_dict()` is an ordered dictionary mapping parameter names → tensors. Saving it (rather than the whole
model object) is the recommended PyTorch way: it's portable and doesn't pickle your class definition.

> **Important:** to load, you must first build an architecture with the **same shape**, then
> `load_state_dict(...)`. The tensors are matched by *name and shape*.

In [ ]:
Path("checkpoints").mkdir(exist_ok=True)
CKPT = "checkpoints/baseline_mlp.pt"

# --- SAVE ---
torch.save(base_model.state_dict(), CKPT)
print("Saved state_dict to", CKPT)
print("Keys:", list(base_model.state_dict().keys())[:4], "...")

# --- LOAD --- (build the same architecture, then load weights)
reloaded = MLP(in_features=N_FEATURES, hidden_sizes=(16,), n_classes=N_CLASSES,
               activation="relu", use_batchnorm=True).to(DEVICE)
reloaded.load_state_dict(torch.load(CKPT, map_location=DEVICE))
reloaded.eval()

# Verify the reloaded model produces identical predictions
with torch.no_grad():
    a = base_model(torch.tensor(X_test, dtype=torch.float32, device=DEVICE)).argmax(1)
    b = reloaded(torch.tensor(X_test,  dtype=torch.float32, device=DEVICE)).argmax(1)
print("Reloaded model reproduces predictions:", bool(torch.equal(a, b)))

---
## 9. Hyperparameter investigation

**Requirement: investigate hyperparameters.** We sweep, one dimension at a time, over:
- **number of hidden layers** (1 vs 2)
- **neurons per hidden layer** (8, 16, 64)
- **learning rate** (1e-4, 1e-3, 1e-2)
- **batch size** (8, 16, 64)
- **activation function** (ReLU, LeakyReLU, Tanh)

Because the dataset is tiny (168 training samples), each run is seconds. `run_experiment()` returns a tidy
summary row so we can build a comparison table.

> **Tip for your write-up:** run this cell, then paste the resulting table into your report and comment on
> which knobs mattered most. The number of *neurons* and the *learning rate* usually move accuracy the most
> on a dataset this small.

In [ ]:
def run_experiment(name, batch_size=16, hidden_sizes=(16,), activation="relu",
                   lr=1e-3, dropout=0.0, weight_decay=0.0, use_batchnorm=True,
                   n_epochs=80, early_stopping_patience=15, verbose=False):
    """Train one configuration and return a summary dict."""
    # Fresh loaders (so batch_size takes effect)
    tr_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    va_loader = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)

    model, opt = make_model(hidden_sizes=hidden_sizes, activation=activation,
                            dropout=dropout, use_batchnorm=use_batchnorm,
                            weight_decay=weight_decay, lr=lr)

    model, hist = train(model, tr_loader, va_loader, opt, n_epochs=n_epochs,
                        logger=Logger(run_name=name, use_tensorboard=False),
                        early_stopping_patience=early_stopping_patience,
                        plot_every=0, verbose=verbose)

    train_acc = evaluate(model, tr_loader)[1]
    val_loss, val_acc = evaluate(model, va_loader)

    return {
        "name": name,
        "hidden_sizes": str(hidden_sizes),
        "activation": activation,
        "lr": lr,
        "batch_size": batch_size,
        "dropout": dropout,
        "weight_decay": weight_decay,
        "val_loss": round(val_loss, 4),
        "val_acc": round(val_acc, 4),
        "train_acc": round(train_acc, 4),
        "epochs_run": len(hist["train_loss"]),
    }


# Design the sweep: each entry varies ONE thing from the baseline.
sweep = []

# --- hidden layers ---
sweep.append(dict(name="1 layer / 16",   hidden_sizes=(16,)))
sweep.append(dict(name="2 layers / 32,16", hidden_sizes=(32, 16)))

# --- neurons per layer ---
sweep.append(dict(name="neurons=8",      hidden_sizes=(8,)))
sweep.append(dict(name="neurons=64",     hidden_sizes=(64,)))

# --- learning rate ---
sweep.append(dict(name="lr=1e-4", lr=1e-4))
sweep.append(dict(name="lr=1e-2", lr=1e-2))

# --- batch size ---
sweep.append(dict(name="batch=8",  batch_size=8))
sweep.append(dict(name="batch=64", batch_size=64))

# --- activation ---
sweep.append(dict(name="activation=leakyrelu", activation="leakyrelu"))
sweep.append(dict(name="activation=tanh",      activation="tanh"))

results = [run_experiment(**cfg) for cfg in sweep]
df_sweep = pd.DataFrame(results).sort_values("val_acc", ascending=False)
df_sweep

In [ ]:
# Bar chart of validation accuracy across the sweep
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(df_sweep["name"], df_sweep["val_acc"], color="tab:blue")
ax.axvline(base_val_acc, color="red", ls="--", label=f"baseline ({base_val_acc:.3f})")
ax.set_xlabel("validation accuracy"); ax.set_xlim(0, 1.05)
ax.set_title("Hyperparameter sweep — validation accuracy")
ax.legend(); plt.tight_layout(); plt.show()

---
## 10. Regularization: compare against the baseline

**Requirement: apply at least one regularization technique and compare with the baseline.**

On a tiny dataset (168 training samples), an MLP can easily **overfit** — training accuracy hits 1.0 while
validation stalls. We compare three techniques:

1. **Dropout** — randomly zeros activations during training, forcing redundant representations.
2. **Weight decay (L2)** — penalises large weights via the optimizer's `weight_decay` argument.
3. **Early stopping** — stop when validation loss stops improving and restore the best weights
   (already built into our `train()` and used at patience=15).

We then plot the **gap** between train and validation loss — a classic overfitting diagnostic.

In [ ]:
reg_results = []

# Baseline is reused from above for a fair comparison.
reg_results.append(dict(variant="baseline (none)", val_acc=round(base_val_acc, 4),
                        train_acc=round(base_train_acc, 4),
                        gap=round(abs(base_train_acc - base_val_acc), 3)))

# --- Dropout ---
r = run_experiment("dropout=0.3", dropout=0.3)
reg_results.append(dict(variant="dropout 0.3", val_acc=r["val_acc"],
                        train_acc=r["train_acc"], gap=round(abs(r["train_acc"]-r["val_acc"]), 3)))

# --- Weight decay (L2) ---
r = run_experiment("weight_decay=1e-2", weight_decay=1e-2)
reg_results.append(dict(variant="weight decay 1e-2", val_acc=r["val_acc"],
                        train_acc=r["train_acc"], gap=round(abs(r["train_acc"]-r["val_acc"]), 3)))

# --- Dropout + weight decay combined ---
r = run_experiment("dropout=0.3 + wd=1e-2", dropout=0.3, weight_decay=1e-2)
reg_results.append(dict(variant="dropout + weight decay", val_acc=r["val_acc"],
                        train_acc=r["train_acc"], gap=round(abs(r["train_acc"]-r["val_acc"]), 3)))

df_reg = pd.DataFrame(reg_results)
df_reg

In [ ]:
# Chart: train vs val accuracy per variant, plus the generalization gap.
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(df_reg))
w = 0.38
ax.bar(x - w/2, df_reg["train_acc"], w, label="train acc", color="tab:orange")
ax.bar(x + w/2, df_reg["val_acc"],   w, label="val acc",   color="tab:blue")
ax.set_xticks(x); ax.set_xticklabels(df_reg["variant"], rotation=15, ha="right")
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.1)
ax.set_title("Regularization comparison vs. baseline")
ax.legend(); plt.tight_layout(); plt.show()

print("Interpretation: a SMALL train–val gap means the model generalises; "
      "a LARGE gap signals overfitting that regularization should reduce.")

---
## 11. Final model → test set → `network.csv`

**Requirement: create `network.csv` with your final predictions of the test set.**

We now:
1. Dictate the final configuration (best from the study — edit to taste).
2. Retrain on the training set, monitor on validation, early-stop.
3. Evaluate **once** on the test set.
4. Save the final `state_dict`.
5. Write `network.csv` containing the test-set predictions.

For the CSV we include the **true class, predicted class, and the probability of the prediction** so the
grader can see how confident the model is.

In [ ]:
# ------------------------- FINAL CONFIGURATION -------------------------
# Edit these to whatever your study found best.
FINAL = dict(hidden_sizes=(32, 16), activation="relu", lr=1e-3,
             batch_size=16, dropout=0.3, weight_decay=1e-2)

final_train_loader = DataLoader(train_ds, batch_size=FINAL["batch_size"], shuffle=True,  num_workers=0)
final_val_loader   = DataLoader(val_ds,   batch_size=FINAL["batch_size"], shuffle=False, num_workers=0)
final_test_loader  = DataLoader(test_ds,  batch_size=FINAL["batch_size"], shuffle=False, num_workers=0)

final_model, final_opt = make_model(
    hidden_sizes=FINAL["hidden_sizes"], activation=FINAL["activation"],
    dropout=FINAL["dropout"], weight_decay=FINAL["weight_decay"], lr=FINAL["lr"],
)

final_logger = Logger(run_name="final")
final_model, final_hist = train(
    final_model, final_train_loader, final_val_loader, final_opt,
    n_epochs=150, logger=final_logger, early_stopping_patience=20, plot_every=25,
)

# ------------------------- SAVE FINAL WEIGHTS -------------------------
torch.save(final_model.state_dict(), "checkpoints/final_mlp.pt")
print("Saved final weights to checkpoints/final_mlp.pt")

In [ ]:
# ------------------------- TEST-SET EVALUATION -------------------------
# Reload from disk to prove the saved weights are what we evaluate.
loaded = MLP(in_features=N_FEATURES, hidden_sizes=FINAL["hidden_sizes"],
             n_classes=N_CLASSES, activation=FINAL["activation"],
             dropout=FINAL["dropout"], use_batchnorm=True).to(DEVICE)
loaded.load_state_dict(torch.load("checkpoints/final_mlp.pt", map_location=DEVICE))
loaded.eval()

# Gather predictions + probabilities on the test set.
all_probs, all_preds, all_true = [], [], []
with torch.no_grad():
    for xb, yb in final_test_loader:
        xb = xb.to(DEVICE)
        logits = loaded(xb)
        probs = torch.softmax(logits, dim=1)     # for reporting only
        all_probs.append(probs.cpu())
        all_preds.append(probs.argmax(dim=1).cpu())
        all_true.append(yb)

all_probs = torch.cat(all_probs).numpy()
all_preds = torch.cat(all_preds).numpy()
all_true  = torch.cat(all_true).numpy()

test_acc = (all_preds == all_true).mean()
test_loss, _ = evaluate(loaded, final_test_loader)
print(f"FINAL TEST  ->  loss {test_loss:.4f} | accuracy {test_acc:.3f}")

In [ ]:
# ------------------------- WRITE network.csv -------------------------
# Human-readable class names (UCI order): Kama=0, Rosa=1, Canadian=2
CLASS_NAMES = ["Kama", "Rosa", "Canadian"]

pred_df = pd.DataFrame({
    "sample":        np.arange(len(all_true)),
    "true_class":    [CLASS_NAMES[i] for i in all_true],
    "predicted_class": [CLASS_NAMES[i] for i in all_preds],
    "pred_confidence": np.round(all_probs.max(axis=1), 4),
    "correct":       (all_preds == all_true).astype(int),
    # also include class probabilities per class, in case the grader wants them
    "p_Kama":     np.round(all_probs[:, 0], 4),
    "p_Rosa":     np.round(all_probs[:, 1], 4),
    "p_Canadian": np.round(all_probs[:, 2], 4),
})

OUT = "network.csv"
pred_df.to_csv(OUT, index=False)
print(f"Wrote {OUT} with {len(pred_df)} test predictions.")
print(f"Test accuracy in this file: {pred_df['correct'].mean():.3f}")

pred_df.head(10)

---
## 12. Wrap-up & things to write up

### What you built
A hand-rolled MLP in PyTorch: a custom `nn.Module` with a `forward()`, a `Dataset` + `DataLoader`,
mini-batch training, a **hand-written cross-entropy loss** normalised by the inputs and accumulated
across mini-batches, an Adam optimizer, BatchNorm, `model.train()` / `model.eval()` loops, `state_dict`
save & load, live loss visualisation, a hyperparameter study, and a regularization comparison.

### Suggested report sections
1. **Architecture** — the 7 → 16 → 3 baseline and why logits (no final softmax).
2. **Loss** — your derivation of cross-entropy and the normalisation argument; show the sanity check
   matching `nn.CrossEntropyLoss`.
3. **Hyperparameters** — paste the sweep table and bar chart; state the best combination and *why*.
4. **Regularization** — the train/val gap table and chart; did dropout/weight-decay help on 168 samples?
5. **Final results** — test loss & accuracy; attach `network.csv`.

### Common pitfalls (learn from them!)
- **Forgetting `model.eval()`** during validation → dropout stays on, BatchNorm uses batch stats, results
  look random.
- **Forgetting `optimizer.zero_grad()`** → gradients accumulate across batches and training diverges.
- **Scaling with test-set statistics** → data leakage. We fit the scaler on train only.
- **Float64 inputs** → `nn.Linear` wants `float32`; our `Dataset` casts explicitly.
- **One-hot targets** → our loss wants integer class indices.

### Extending the assignment
- Add a **learning-rate scheduler** (`torch.optim.lr_scheduler.StepLR`) — already plumbed into `train()`.
- Try **k-fold cross-validation** instead of a single 10% validation split (210 samples is small).
- Compare Adam against plain `SGD(momentum=0.9)`.
- Log the **confusion matrix** on the test set for a richer picture than accuracy alone.

---
**Good luck!** Remember: change the `FINAL` config to your best-found settings, re-run the last two cells,
and you're done. 🎓